# 13 — Bootstrap Mode

Generate synthetic data by bootstrapping (sampling with replacement) from real data. Provides a baseline with exact distribution preservation.

In [ ]:
from sqllocks_spindle import Spindle
from sqllocks_spindle.inference.tier3_research import BootstrapMode
from sqllocks_spindle.inference.comparator import FidelityReport
import importlib
from sqllocks_spindle.cli import _get_domain_registry

spindle = Spindle()
reg = _get_domain_registry()
mod, cls, _ = reg['retail']
domain = getattr(importlib.import_module(mod), cls)(schema_mode='3nf')
reference = spindle.generate(domain=domain, scale='small', seed=42)
table = next(iter(reference.tables))
real_df = reference.tables[table]
print(f'Source: {table}  rows={len(real_df)}')

In [ ]:
# Bootstrap: identical distribution
bm = BootstrapMode(add_jitter=True, jitter_std_fraction=0.01)
synth_df, boot_result = bm.generate(real_df, n_rows=len(real_df), seed=42)
print(f'Generated: {boot_result.n_rows} rows from {boot_result.source_rows_used} source rows')

In [ ]:
# Fidelity of bootstrap vs spindle generation
try:
    from sqllocks_spindle.inference.comparator import FidelityReport
    boot_report = FidelityReport.score(real_df, synth_df, table_name=table)
    spindle_gen = spindle.generate(domain=domain, scale='small', seed=99)
    spindle_report = FidelityReport.score(real_df, spindle_gen.tables[table], table_name=table)
    print(f'Bootstrap fidelity: {boot_report.overall_score:.1f}/100')
    print(f'Spindle fidelity:   {spindle_report.overall_score:.1f}/100')
    print('(Bootstrap preserves exact distributions by design)')
except Exception as e:
    print(f'Fidelity comparison skipped: {e}')